# Notebook 01 — Data Extraction

**Project:** VC Investments Analytics  
**Phase:** Week 1 — Data Sourcing & ETL  
**Objective:** Load the raw dataset, perform an initial structural audit, and commit it to `data/raw/` unchanged.

> ⚠️ **Rule:** The file saved in `data/raw/` must never be edited. All transformations happen downstream in `02_cleaning.ipynb`.

## 1. Imports & Configuration

In [15]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────
RAW_SOURCE = '../data/raw/investments_VC.csv'   # dataset already placed here
# If running from a different location, update RAW_SOURCE accordingly.

print('Pandas version:', pd.__version__)
print('NumPy  version:', np.__version__)

Pandas version: 3.0.2
NumPy  version: 2.4.4


## 2. Load Raw Dataset

The file contains non-UTF-8 characters (e.g. degree symbols in city names), so `encoding='latin-1'` is required.

In [16]:
df_raw = pd.read_csv(RAW_SOURCE, encoding='latin-1')

print(f'Rows : {df_raw.shape[0]:,}')
print(f'Cols : {df_raw.shape[1]}')
print()
print('Column names:')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i:02d}] "{col}"')

Rows : 54,294
Cols : 39

Column names:
  [00] "permalink"
  [01] "name"
  [02] "homepage_url"
  [03] "category_list"
  [04] " market "
  [05] " funding_total_usd "
  [06] "status"
  [07] "country_code"
  [08] "state_code"
  [09] "region"
  [10] "city"
  [11] "funding_rounds"
  [12] "founded_at"
  [13] "founded_month"
  [14] "founded_quarter"
  [15] "founded_year"
  [16] "first_funding_at"
  [17] "last_funding_at"
  [18] "seed"
  [19] "venture"
  [20] "equity_crowdfunding"
  [21] "undisclosed"
  [22] "convertible_note"
  [23] "debt_financing"
  [24] "angel"
  [25] "grant"
  [26] "private_equity"
  [27] "post_ipo_equity"
  [28] "post_ipo_debt"
  [29] "secondary_market"
  [30] "product_crowdfunding"
  [31] "round_A"
  [32] "round_B"
  [33] "round_C"
  [34] "round_D"
  [35] "round_E"
  [36] "round_F"
  [37] "round_G"
  [38] "round_H"


In [17]:
# Quick visual of the first 5 rows
df_raw.head()

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Structural Audit

In [18]:
# --- 3a. Data types ---
print('=== Data Types ===')
print(df_raw.dtypes.to_string())

=== Data Types ===
permalink                   str
name                        str
homepage_url                str
category_list               str
 market                     str
 funding_total_usd          str
status                      str
country_code                str
state_code                  str
region                      str
city                        str
funding_rounds          float64
founded_at                  str
founded_month               str
founded_quarter             str
founded_year            float64
first_funding_at            str
last_funding_at             str
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding 

In [19]:
# --- 3b. Missing value summary ---
missing = df_raw.isnull().sum().rename('missing_count')
missing_pct = (df_raw.isnull().mean() * 100).round(2).rename('missing_%')
audit = pd.concat([missing, missing_pct], axis=1)
print('=== Missing Values ===')
print(audit[audit['missing_count'] > 0].to_string())

=== Missing Values ===
                      missing_count  missing_%
permalink                      4856       8.94
name                           4857       8.95
homepage_url                   8305      15.30
category_list                  8817      16.24
 market                        8824      16.25
 funding_total_usd             4856       8.94
status                         6170      11.36
country_code                  10129      18.66
state_code                    24133      44.45
region                        10129      18.66
city                          10972      20.21
funding_rounds                 4856       8.94
founded_at                    15740      28.99
founded_month                 15812      29.12
founded_quarter               15812      29.12
founded_year                  15812      29.12
first_funding_at               4856       8.94
last_funding_at                4856       8.94
seed                           4856       8.94
venture                        4856  

In [20]:
# --- 3c. Fully-null rows (empty rows that crept in during export) ---
fully_null = df_raw.isnull().all(axis=1).sum()
print(f'Fully-null rows (all columns = NaN): {fully_null:,}')
print(f'These represent {fully_null / len(df_raw) * 100:.1f}% of total rows.')

Fully-null rows (all columns = NaN): 4,856
These represent 8.9% of total rows.


In [21]:
# --- 3d. Duplicate permalinks ---
dup_count = df_raw['permalink'].duplicated(keep=False).sum()
print(f'Rows with a duplicated permalink: {dup_count:,}')

Rows with a duplicated permalink: 4,860


In [22]:
# --- 3e. Cardinality check on key categoricals ---
cat_cols = ['status', 'country_code', ' market ', ' market '.strip()]
# Use raw column names as-is
for col in [' market ', 'status', 'country_code']:
    print(f'{col!r:30s}  unique={df_raw[col].nunique():5d}   null={df_raw[col].isnull().sum():5,}')

' market '                      unique=  753   null=8,824
'status'                        unique=    3   null=6,170
'country_code'                  unique=  115   null=10,129


In [23]:
# --- 3f. Inspect the funding_total_usd column closely ---
# It is stored as a string with locale-formatted numbers and
# uses '-' as a placeholder for missing/zero values.
print('Sample values of " funding_total_usd ":',
      df_raw[' funding_total_usd '].dropna().head(10).tolist())
print()
dash_count = (df_raw[' funding_total_usd '].str.strip() == '-').sum()
print(f'  Rows with dash placeholder: {dash_count:,}')

Sample values of " funding_total_usd ": [' 17,50,000 ', ' 40,00,000 ', ' 40,000 ', ' 15,00,000 ', ' 60,000 ', ' 70,00,000 ', ' 49,12,393 ', ' 20,00,000 ', ' -   ', ' 41,250 ']

  Rows with dash placeholder: 8,531


In [24]:
# --- 3g. Column names with leading / trailing whitespace ---
padded = [c for c in df_raw.columns if c != c.strip()]
print('Column names with surrounding whitespace:', padded)

Column names with surrounding whitespace: [' market ', ' funding_total_usd ']


In [25]:
# --- 3h. Status distribution ---
print('Status value counts:')
print(df_raw['status'].value_counts(dropna=False))

Status value counts:
status
operating    41829
NaN           6170
acquired      3692
closed        2603
Name: count, dtype: int64


In [26]:
# --- 3i. Founding year range ---
print('founded_year  min:', df_raw['founded_year'].min())
print('founded_year  max:', df_raw['founded_year'].max())
print('Pre-1990 count  :', (df_raw['founded_year'] < 1990).sum())

founded_year  min: 1902.0
founded_year  max: 2014.0
Pre-1990 count  : 998


In [27]:
# --- 3j. Numeric funding columns overview ---
funding_cols = ['seed', 'venture', 'angel', 'grant', 'round_A', 'round_B',
                'round_C', 'round_D', 'round_E', 'round_F']
df_raw[funding_cols].describe().round(0)

,seed,venture,angel,grant,round_A,round_B,round_C,round_D,round_E,round_F
count,49438.0,4.943800e+04,49438.0,49438.0,49438.0,49438.0,49438.0,4.943800e+04,49438.0,4.943800e+04
mean,217321.0,7.501051e+06,65419.0,162845.0,1243955.0,1492891.0,1205356.0,7.375260e+05,342468.0,1.697690e+05
std,1056985.0,2.847112e+07,658291.0,5612088.0,5531974.0,7472704.0,7993592.0,9.815218e+06,5406915.0,6.277905e+06
min,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00
25%,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00
50%,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00
75%,25000.0,5.000000e+06,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.000000e+00
max,130000000.0,2.351000e+09,63590263.0,750500000.0,319000000.0,542000000.0,490000000.0,1.200000e+09,400000000.0,1.060000e+09


## 4. Key Issues Identified

| # | Issue | Column(s) | Action (in 02_cleaning) |
|---|-------|-----------|-------------------------|
| 1 | Leading/trailing whitespace in column names | `' market '`, `' funding_total_usd '` | Strip all column names |
| 2 | 4,856 fully-null rows (CSV export artefact) | All | Drop with `dropna(how='all')` |
| 3 | 2 duplicate `permalink` entries | `permalink` | Deduplicate on `permalink` |
| 4 | `funding_total_usd` stored as string with commas and dash placeholders | `funding_total_usd` | Remove formatting, replace `-` with NaN, cast to float |
| 5 | Date columns stored as plain strings | `founded_at`, `first_funding_at`, `last_funding_at` | Parse with `pd.to_datetime` |
| 6 | `category_list` wrapped in leading/trailing pipe characters | `category_list` | Strip outer `|` |
| 7 | Funding round columns (`seed`, `venture`, …, `round_H`) have NaN for non-participating rounds | All funding round cols | Fill NaN → 0 |
| 8 | `market` values have surrounding whitespace | `market` | Strip `.str.strip()` |

## 5. Initial Data Dictionary

| Column | Type (raw) | Description |
|--------|-----------|-------------|
| `permalink` | string | Unique identifier / URL slug for the startup |
| `name` | string | Company name |
| `homepage_url` | string | Company website URL |
| `category_list` | string | Pipe-delimited list of product/market categories |
| `market` | string | Primary market segment |
| `funding_total_usd` | string (needs cast) | Total funding raised in USD (comma-formatted) |
| `status` | string | Company status: `operating`, `acquired`, `closed` |
| `country_code` | string | 3-letter ISO country code |
| `state_code` | string | US state code (null for non-US) |
| `region` | string | Geographic region |
| `city` | string | City of headquarters |
| `funding_rounds` | float | Number of funding rounds completed |
| `founded_at` | string (needs parse) | Company founding date (`YYYY-MM-DD`) |
| `founded_month` | string | Month of founding (`YYYY-MM`) |
| `founded_quarter` | string | Quarter of founding (`YYYY-QN`) |
| `founded_year` | float | Year of founding |
| `first_funding_at` | string (needs parse) | Date of first funding event |
| `last_funding_at` | string (needs parse) | Date of most recent funding event |
| `seed` | float | Seed round amount (USD) |
| `venture` | float | Total venture funding (USD) |
| `equity_crowdfunding` | float | Equity crowdfunding amount (USD) |
| `undisclosed` | float | Undisclosed funding amount (USD) |
| `convertible_note` | float | Convertible note amount (USD) |
| `debt_financing` | float | Debt financing amount (USD) |
| `angel` | float | Angel investment amount (USD) |
| `grant` | float | Grant amount (USD) |
| `private_equity` | float | Private equity amount (USD) |
| `post_ipo_equity` | float | Post-IPO equity amount (USD) |
| `post_ipo_debt` | float | Post-IPO debt amount (USD) |
| `secondary_market` | float | Secondary market transaction amount (USD) |
| `product_crowdfunding` | float | Product crowdfunding amount (USD) |
| `round_A` – `round_H` | float | Individual round amounts by series (A through H) |

## 6. Confirm Raw File Location

The raw file has been placed at `data/raw/investments_VC.csv`.  
We verify it is accessible and print its size for the record.

In [28]:
file_size_mb = os.path.getsize(RAW_SOURCE) / (1024 ** 2)
print(f'Raw file path : {os.path.abspath(RAW_SOURCE)}')
print(f'File size     : {file_size_mb:.2f} MB')
print(f'Row count     : {df_raw.shape[0]:,}')
print(f'Column count  : {df_raw.shape[1]}')
print()
print('✅  Extraction complete. Proceed to 02_cleaning.ipynb.')

Raw file path : /Users/krishnaverma/Desktop/A_G11_StartupInvestments/data/raw/investments_VC.csv
File size     : 11.95 MB
Row count     : 54,294
Column count  : 39

✅  Extraction complete. Proceed to 02_cleaning.ipynb.
